In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1 How many lesson pages are in the dataset? (Ans = 72)

In [6]:
print("Number of pages =",len(documents))

Number of pages = 72


### Q2. Indexing and searching (answer: 01-agentic-rag/lessons/14-agentic-loop.md)

In [7]:
from minsearch import Index

In [8]:
# Index the documents with minsearch - make content a text field and filename a keyword 
# field. Then search with this query:

# How does the agentic loop keep calling the model until it stops?

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)


In [10]:
search_results = index.search('How does the agentic loop keep calling the model until it stops?')
print(search_results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


### Q3. RAG

In [ ]:

from openai import OpenAI

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course="llm-zoomcamp",
        model="gpt-5.4-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model


    def search(self, query, num_results=5):
        filter_dict = {"course": self.course}

        return self.index.search(
            query,
            num_results=num_results,
            filter_dict=filter_dict
        )
    

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["section"])
            lines.append("Q: " + doc["question"])
            lines.append("A: " + doc["answer"])
            lines.append("")

        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )
    
    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text
    
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

In [14]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the course notes.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [15]:
assistant.search('How does the agentic loop keep calling the model until it stops?')

FilterValidationError: Unknown filter field 'course'. Valid fields are: ['filename']